In [ ]:
# ==============================================================================
# BLOCK 1 — INSTALL & IMPORTS
# Water Surface Debris Annotation Tool
# Run this cell ONCE at the start of every Colab session
# ==============================================================================

# ── System / Drive ─────────────────────────────────────────────────────────────
from google.colab import drive
import os, json, re
from pathlib import Path

# ── Image handling ─────────────────────────────────────────────────────────────
from PIL import Image
import numpy as np
import base64
from io import BytesIO

# ── Colab display / widgets ────────────────────────────────────────────────────
from IPython.display import display, HTML, Javascript, clear_output
import ipywidgets as widgets

# ── Mount Google Drive ─────────────────────────────────────────────────────────
drive.mount('/content/drive', force_remount=False)

print("✅  All libraries ready.")
print("✅  Google Drive mounted at /content/drive")

Mounted at /content/drive
✅  All libraries ready.
✅  Google Drive mounted at /content/drive


In [ ]:
# ==============================================================================
# BLOCK 2 — CONFIGURATION
# Set your Drive folder paths here ONLY — nothing else changes
# ==============================================================================

# ── INPUT: folder containing your .jpg/.png images ────────────────────────────
IMAGE_FOLDER  = "/content/drive/MyDrive/Fydpwork/TestImg"   # ← change this

# ── OUTPUT: folder where annotation JSONs will be saved ───────────────────────
JSON_FOLDER   = "/content/drive/MyDrive/Fydpwork/TestJson"    # ← change this

# ── Supported image extensions ─────────────────────────────────────────────────
VALID_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# ── Coordinate format ──────────────────────────────────────────────────────────
# Bounding box stored as: [ymin, xmin, ymax, xmax]  (0-1000 normalized)
COORD_FORMAT  = "ymin_xmin_ymax_xmax_norm1000"

# ── Validation ─────────────────────────────────────────────────────────────────
def _natural_key(fname):
    """Sort 'p_img_1, p_img_2, ..., p_img_10' in numeric order, not lexical."""
    return [int(tok) if tok.isdigit() else tok.lower()
            for tok in re.split(r'(\d+)', fname)]

def validate_config():
    errors = []
    if not os.path.isdir(IMAGE_FOLDER):
        errors.append(f"❌  IMAGE_FOLDER not found: {IMAGE_FOLDER}")
    os.makedirs(JSON_FOLDER, exist_ok=True)          # auto-create output folder
    if errors:
        for e in errors: print(e)
        raise FileNotFoundError("Fix paths above, then re-run this cell.")

    images = sorted(
        [f for f in os.listdir(IMAGE_FOLDER)
         if Path(f).suffix.lower() in VALID_EXT],
        key=_natural_key
    )
    print(f"✅  Config valid.")
    print(f"📂  Images found : {len(images)}")
    print(f"💾  JSON output  : {JSON_FOLDER}")
    return images

IMAGE_LIST = validate_config()

❌  IMAGE_FOLDER not found: /content/drive/MyDrive/Fydpwork/TestImg


FileNotFoundError: Fix paths above, then re-run this cell.

In [ ]:
# ==============================================================================
# BLOCK 3 — ANNOTATION ENGINE  v7.1  (click-reduction & readability pass)
#
# Changes vs v6.5:
#   - REMOVED the "lock canvas until you confirm lighting" step. The canvas is
#     active the instant an image loads — draw immediately, zero setup clicks.
#   - REMOVED the "Save & Next?" Yes/No confirmation dialog. Clicking
#     "Save & Next Image" (or pressing Enter) saves and advances in one click.
#   - Environmental metadata (lighting/water body/coverage/turbidity/etc.) now
#     PERSISTS from image to image instead of resetting every time — since
#     these rarely change between consecutive photos, annotators usually
#     touch nothing and just move on. Per-image JSON overrides still apply
#     if you jump back to an already-annotated image.
#   - NEW: hold Shift while drawing a box to instantly tag it with the
#     LAST class you used — skips the class picker entirely (0 clicks).
#   - NEW: number-key shortcuts (1-9, 0) in the class picker, so most boxes
#     need one drag + one keystroke instead of one drag + one click-hunt.
#   - NEW: Enter = Save & Next, U = Undo Last (ignored while typing in a
#     text field so it never interferes with Notes).
#   - Bigger fonts, bigger dropdowns/buttons/textarea, bigger turbidity
#     swatches, and a thicker, higher-contrast progress bar.
# ==============================================================================

import json, os
from pathlib import Path
from PIL import Image, ImageOps
from io import BytesIO
import base64
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output, Javascript

# ══════════════════════════════════════════════════════════════════════════════
# MODULE A — TAXONOMY + COLOURS
# ══════════════════════════════════════════════════════════════════════════════
TAXONOMY = {
    "plastic_bottle": {
        "display": "🥤 Plastic Bottle",
        "fcot": {
            "Primary Cue": "Rigid curved body with a visible neck or cap, holding its 3D shape on the water surface.",
            "Observation": "Bottle-shaped floating object with smooth glossy surfaces and a consumer colour palette (transparent, white, blue, green, or brown). Label band may be intact, peeled, or absent. Body may be intact, crushed, or dented.",
            "Contrastive Rules": [
                "Unlike polythene, plastic_bottle holds a rigid 3D shape instead of draping flat.",
                "Unlike foam_waste, plastic_bottle has a glossy curved surface instead of a matte crumbly one.",
                "Unlike solid_waste, plastic_bottle is a single identifiable bottle, not a mixed cluster."
            ],
            "Static-Frame Disambiguation": "The bottle outline follows the object's curved body with a hard edge; a bright spot from sky reflection has a soft gradient that does not trace any object boundary.",
            "Decision Rule": "Assign plastic_bottle when a rigid curved body shows a neck taper or a cap, even if the label is missing.",
            "Fallback Rule": "If less than half visible, commit on any one of: visible cap, glossy curved edge segment, or cylindrical outline.",
            "Failure Mode": "Most often confused with polythene when the bottle is crushed flat — tiebreaker is any visible rigid curve or neck remnant.",
            "Instance Note": "",
            "Conclusion": "Class: plastic_bottle. Single-item consumer macroplastic with persistent floating behaviour."
        }
    },
    "polythene": {
        "display": "🛍️ Polythene",
        "fcot": {
            "Primary Cue": "Thin flexible translucent film that folds, wrinkles, or drapes directly along the water surface.",
            "Observation": "Flat or partially submerged sheet-like plastic with soft irregular edges and visible folds. Usually transparent, white, black, or printed. In dark or turbid water appears as a lighter translucent patch against the darker background.",
            "Contrastive Rules": [
                "Unlike plastic_bottle, polythene does not maintain a rigid 3D structure.",
                "Unlike fabric_waste, polythene has smooth stretched edges instead of soft frayed fibres.",
                "Unlike foam_waste, polythene is thin and flexible instead of thick and rigid."
            ],
            "Static-Frame Disambiguation": "The bag has folded edges and a translucent body with hard outline; a water glare patch has soft spreading edges and no folds.",
            "Decision Rule": "Assign polythene when a thin draping translucent film is visible with no weave pattern and no rigid form.",
            "Fallback Rule": "If folded into a narrow strip or nearly submerged, commit on translucency, drape conforming to water, and faint folded outline.",
            "Failure Mode": "Most often confused with fabric_waste under low resolution — tiebreaker is glossy translucent surface versus matte fibrous surface.",
            "Instance Note": "",
            "Conclusion": "Class: polythene. Flexible thin-film plastic debris with high entanglement risk."
        }
    },
    "foam_waste": {
        "display": "🧊 Foam Waste",
        "fcot": {
            "Primary Cue": "Bright matte-white rigid material with granular or crumbly broken edges, floating very high on the water surface.",
            "Observation": "White or yellowed floating block, slab, or fragment with irregular broken boundaries. Surface reflects light diffusely instead of producing sharp highlights.",
            "Contrastive Rules": [
                "Unlike polythene, foam_waste maintains a thick rigid form.",
                "Unlike plastic_bottle, foam_waste lacks glossy curved reflections.",
                "Unlike algal_bloom, foam_waste forms a solid object with clear physical boundaries."
            ],
            "Static-Frame Disambiguation": "Foam has a hard physical edge that follows the block outline; a bright sky reflection has a soft gradient that spreads without any solid boundary.",
            "Decision Rule": "Assign foam_waste when a matte-white rigid object with crumbly broken edges floats above the waterline.",
            "Fallback Rule": "If only a corner is visible, commit on matte-white colour and crumbly broken edge geometry.",
            "Failure Mode": "Most often confused with bright sky reflections — tiebreaker is real foam retains a hard physical outline.",
            "Instance Note": "",
            "Conclusion": "Class: foam_waste. High-fragmentation plastic material with elevated microplastic risk."
        }
    },
    "water_hyacinth": {
        "display": "🌿 Water Hyacinth",
        "fcot": {
            "Primary Cue": "Rounded glossy green leaves arranged in floating rosettes above the water on bulbous inflated stems.",
            "Observation": "Floating aquatic plant with bulbous leaf bases and clustered leaf rosettes. Healthy leaves are glossy green; older plants show yellow edges. Often forms dense mats.",
            "Contrastive Rules": [
                "Unlike algal_bloom, water_hyacinth has individual leaves and plant structures visible.",
                "Unlike organic_waste, water_hyacinth is living and structurally intact.",
                "Unlike shoreline vegetation, water_hyacinth floats freely with no rooted stem."
            ],
            "Static-Frame Disambiguation": "Real water_hyacinth has volumetric depth with a cast shadow; a reflection of shoreline vegetation is inverted and flat.",
            "Decision Rule": "Assign water_hyacinth when intact rounded glossy leaves or inflated bulbous stems are visible.",
            "Fallback Rule": "If partially occluded, commit on intact rounded glossy leaves or inflated stem shapes.",
            "Failure Mode": "Most often confused with organic_waste when damaged — tiebreaker is any intact rosette structure.",
            "Instance Note": "",
            "Conclusion": "Class: water_hyacinth. Invasive floating aquatic vegetation."
        }
    },
    "algal_bloom": {
        "display": "🟢 Algal Bloom",
        "fcot": {
            "Primary Cue": "Diffuse coloured surface film covering a broad continuous water area without solid edges or raised structure.",
            "Observation": "Surface discolouration across a large water area. Bright green = chlorophyta, blue-green/teal = cyanobacteria, red-brown = dinoflagellates. Boundaries fade gradually.",
            "Contrastive Rules": [
                "Unlike water_hyacinth, algal_bloom has no visible leaves, stems, or rosette structures.",
                "Unlike industrial_waste, algal_bloom shows matte single-hued colour rather than iridescent bands.",
                "Unlike organic_waste, algal_bloom is a flat surface film with no solid fragments."
            ],
            "Static-Frame Disambiguation": "A real bloom holds its colour with a soft outer boundary; a reflection of green vegetation is inverted and tied to shoreline features.",
            "Decision Rule": "Assign algal_bloom when a diffuse uniform colour film covers a large continuous region with no solid debris inside.",
            "Fallback Rule": "In low light or turbid water, commit only when green or teal discolouration visibly dominates a large region.",
            "Failure Mode": "Most often confused with reflected vegetation — tiebreaker is bloom shows cool green or teal tones with diffuse soft boundaries.",
            "Instance Note": "",
            "Conclusion": "Class: algal_bloom. Surface eutrophication indicator."
        }
    },
    "organic_waste": {
        "display": "🍂 Organic Waste",
        "fcot": {
            "Primary Cue": "Amorphous floating accumulation of decomposing organic fragments without intact plant rosettes or synthetic items.",
            "Observation": "Pile of leaves, twigs, stems, and decaying plant fragments. Colours range from green-yellow when fresh to dark brown or black when necrotic.",
            "Contrastive Rules": [
                "Unlike water_hyacinth, organic_waste lacks intact glossy leaf rosettes or inflated stems.",
                "Unlike solid_waste, organic_waste contains only natural material.",
                "Unlike wood_debris, organic_waste consists of many fragmented pieces not one solid structure."
            ],
            "Static-Frame Disambiguation": "Organic clumps hold an irregular outline with a cast shadow; reflected vegetation is inverted and flat with no shadow.",
            "Decision Rule": "Assign organic_waste when the floating mass is entirely natural with no man-made items visible inside.",
            "Fallback Rule": "If small synthetic fragments are deeply embedded and visually minor, retain organic_waste.",
            "Failure Mode": "Most often confused with damaged water_hyacinth — tiebreaker is absence of any intact rosette or bulbous stem.",
            "Instance Note": "",
            "Conclusion": "Class: organic_waste. Natural floating organic accumulation undergoing decomposition."
        }
    },
    "solid_waste": {
        "display": "🗑️ Solid Waste",
        "fcot": {
            "Primary Cue": "Cluster of two or more man-made items visibly grouped together on the water surface or at the water edge.",
            "Observation": "Mixed accumulation of identifiable consumer items — wrappers, packaging, cans, broken bricks, ceramic fragments, or food containers.",
            "Contrastive Rules": [
                "Unlike single-item classes, solid_waste contains multiple different debris items grouped together.",
                "Unlike organic_waste, solid_waste contains visible man-made items.",
                "Unlike industrial_waste, solid_waste is composed of solid discrete objects not liquid discharge."
            ],
            "Static-Frame Disambiguation": "Discrete man-made item outlines remain visible with hard geometric edges; reflections of objects are inverted and lack solid outlines.",
            "Decision Rule": "Assign solid_waste when two or more visually distinct man-made items are clustered within one object-width of each other.",
            "Fallback Rule": "If only partial items visible, commit when at least two visually distinct man-made forms can be identified.",
            "Failure Mode": "Most often confused with organic_waste when synthetics are mixed into decaying matter — tiebreaker is any clearly man-made item.",
            "Instance Note": "",
            "Conclusion": "Class: solid_waste. Mixed consumer or construction debris cluster."
        }
    },
    "wood_debris": {
        "display": "🪵 Wood Debris",
        "fcot": {
            "Primary Cue": "Elongated rigid piece of wood with visible grain, bark, or natural branch geometry floating on the water.",
            "Observation": "Branch, plank, log, or bamboo fragment floating horizontally. May show cracks, bark remnants, knots, or sawn ends. Colour ranges from tan-brown to weathered silver-grey.",
            "Contrastive Rules": [
                "Unlike organic_waste, wood_debris forms a single coherent solid structure.",
                "Unlike solid_waste, wood_debris is natural-origin with visible grain or bark texture.",
                "Unlike anchored pilings, wood_debris is floating horizontally not standing vertically."
            ],
            "Static-Frame Disambiguation": "A real wood piece casts a shadow and shows visible grain; a reflected branch is inverted and has no shadow.",
            "Decision Rule": "Assign wood_debris when grain, bark, or branch geometry is visible on a single elongated solid piece.",
            "Fallback Rule": "If mostly submerged or covered by film, commit on any visible bark texture, grain lines, or tapered geometry.",
            "Failure Mode": "Most often confused with anchored pilings — tiebreaker is wood_debris floats horizontally.",
            "Instance Note": "",
            "Conclusion": "Class: wood_debris. Natural or processed wooden debris."
        }
    },
    "industrial_waste": {
        "display": "🛢️ Industrial Waste",
        "fcot": {
            "Primary Cue": "Iridescent rainbow sheen, unnatural strong colour tinting, or frothy chemical foam on the water surface.",
            "Observation": "Thin metallic film with shifting blue, purple, and orange bands; or unnaturally strong colour tinting; or persistent frothy foam near drainage outlets.",
            "Contrastive Rules": [
                "Unlike algal_bloom, industrial_waste shows iridescent rainbow bands not matte single-hued film.",
                "Unlike foam_waste, industrial_waste is liquid-origin froth without a rigid solid block.",
                "Unlike organic_waste, industrial_waste has no leaf or twig fragments."
            ],
            "Static-Frame Disambiguation": "An iridescent sheen has colour shifts following the water film with no solid edge; sky reflection has soft gradient brightness with no colour banding.",
            "Decision Rule": "Assign industrial_waste when iridescent rainbow sheen, unnatural deep colour tinting, or persistent chemical froth covers a continuous area.",
            "Fallback Rule": "Commit only when iridescent banding or unnatural colour saturation is clearly distinguishable from natural water tones.",
            "Failure Mode": "Most often confused with algal_bloom in low light — tiebreaker is industrial_waste shows iridescent rainbow colour shifts.",
            "Instance Note": "",
            "Conclusion": "Class: industrial_waste. Chemical discharge or oil sheen."
        }
    },
    "fabric_waste": {
        "display": "🧵 Fabric Waste",
        "fcot": {
            "Primary Cue": "Flexible cloth or fabric piece with soft frayed edges and a matte draping surface conforming to the water.",
            "Observation": "Cloth, sack, garment, or sari piece with matte fibrous texture. May contain printed patterns, stripes, or faded dye colours.",
            "Contrastive Rules": [
                "Unlike polythene, fabric_waste has fibrous frayed edges and a matte surface.",
                "Unlike wood_debris, fabric_waste is flexible and conforms to the water surface.",
                "Unlike solid_waste, fabric_waste is a single identifiable cloth piece."
            ],
            "Static-Frame Disambiguation": "Real fabric holds its printed pattern with a hard outline; a reflected colour patch has no pattern and no hard outline.",
            "Decision Rule": "Assign fabric_waste when fibrous frayed edges, a printed pattern, or cloth-like drape is visible on a flexible piece.",
            "Fallback Rule": "If tightly folded or partially waterlogged, commit on visible fibrous edges, printed pattern, or matte draping.",
            "Failure Mode": "Most often confused with polythene — tiebreaker is fabric is matte and fibrous; polythene is glossy and translucent.",
            "Instance Note": "",
            "Conclusion": "Class: fabric_waste. Cloth or garment debris."
        }
    },
    "__others__": {
        "display": "✏️ Others (custom)",
        "fcot": {
            "Primary Cue": "", "Observation": "",
            "Contrastive Rules": ["", "", ""],
            "Static-Frame Disambiguation": "", "Decision Rule": "",
            "Fallback Rule": "", "Failure Mode": "",
            "Instance Note": "", "Conclusion": ""
        }
    }
}

DROPDOWN_OPTIONS = [(v["display"], k) for k, v in TAXONOMY.items()]

CLASS_COLORS = {
    "plastic_bottle":  "#00e5ff",
    "polythene":       "#ff4081",
    "foam_waste":      "#90caf9",
    "water_hyacinth":  "#76ff03",
    "algal_bloom":     "#1de9b6",
    "organic_waste":   "#ffab40",
    "solid_waste":     "#ff5252",
    "wood_debris":     "#bcaaa4",
    "industrial_waste":"#e040fb",
    "fabric_waste":    "#ffd740",
    "__others__":      "#b0bec5",
}
DEFAULT_COLOR = "#00e5ff"

# Keyboard shortcuts: 1..9, then 0, assigned in taxonomy order (first 10 classes).
# "__others__" gets the "O" key instead since it doesn't fit 1-9-0.
_SHORTCUT_KEYS = ["1","2","3","4","5","6","7","8","9","0"]

def _build_classes_js():
    entries = []
    shortcut_i = 0
    for k, v in TAXONOMY.items():
        if k == "__others__":
            key = "o"
        else:
            key = _SHORTCUT_KEYS[shortcut_i] if shortcut_i < len(_SHORTCUT_KEYS) else ""
            shortcut_i += 1
        entries.append({
            "key": k, "display": v["display"],
            "color": CLASS_COLORS.get(k, DEFAULT_COLOR),
            "shortcut": key
        })
    return entries

CLASSES_JS = json.dumps(_build_classes_js())

LIGHTING_OPTIONS   = [("☁  Overcast","Overcast"),("🌅 Golden Hour","Golden Hour"),
                      ("☀  Mid-day","Mid-day"),("🌑 Low-light","Low-light")]
WATER_BODY_OPTIONS = [("🪨 Stagnant Pond","Stagnant Pond"),("🌊 Slow-moving Canal","Slow-moving Canal"),
                      ("💧 River","River"),("🏙 Urban Drainage","Urban Drainage"),("🏞 Lake","Lake")]
COVERAGE_OPTIONS   = [("░  0-25%","0-25%"),("▒  25-50%","25-50%"),
                      ("▓  50-75%","50-75%"),("█  75-100%","75-100%")]
TURBIDITY_OPTIONS  = [(str(i), i) for i in range(11)]
REMEDIATION_OPTIONS= [("None","None"),("Low","Low"),("Medium","Medium"),
                      ("High","High"),("Very High","Very High")]

TURBIDITY_SWATCHES = {
    1:  "#fffffd", 2:  "#fefde7", 3:  "#ebe6ca", 4:  "#d9cb9e", 5:  "#c8af6e",
    6:  "#b08b48", 7:  "#9e733d", 8:  "#644b28", 9:  "#3d2c17", 10: "#1f0f02",
}
TURBIDITY_CAPTIONS = {
    1: "Healthy conditions for fish.", 2: "Healthy conditions for fish.", 3: "Healthy conditions for fish.",
    4: "Water starts to lose oxygen; fish find it harder to breathe.",
    5: "Water starts to lose oxygen; fish find it harder to breathe.",
    6: "Water starts to lose oxygen; fish may suffer stunted growth.",
    7: "Fish gills get clogged, can't find food, begin to suffocate.",
    8: "Fish gills get clogged, can't find food, begin to suffocate.",
    9: "Fish gills get clogged, can't find food, begin to suffocate.",
    10: "Fish gills get clogged, can't find food, begin to suffocate.",
}

def derive_primary_pollutant(instances):
    if not instances: return "None"
    from collections import Counter
    return Counter(i["object_name"] for i in instances).most_common(1)[0][0]


# ══════════════════════════════════════════════════════════════════════════════
# MODULE B — IMAGE LOADER
# ══════════════════════════════════════════════════════════════════════════════
class ImageLoader:
    def __init__(self, folder, file_list):
        self.folder, self.file_list = folder, file_list

    def load(self, idx):
        fname  = self.file_list[idx]
        img    = Image.open(os.path.join(self.folder, fname))
        img    = ImageOps.exif_transpose(img)  # fix phone-camera rotation
        img    = img.convert("RGB")
        ow, oh = img.size
        ratio  = min(1000/ow, 1.0)   # slightly bigger canvas -> easier/more precise clicks
        dw, dh = int(ow*ratio), int(oh*ratio)
        buf    = BytesIO()
        img.resize((dw,dh), Image.LANCZOS).save(buf, format="JPEG", quality=92)
        b64    = base64.b64encode(buf.getvalue()).decode()
        return fname, b64, dw, dh, ow, oh


# ══════════════════════════════════════════════════════════════════════════════
# MODULE C — JSON EXPORTER
# ══════════════════════════════════════════════════════════════════════════════
class JSONExporter:
    def __init__(self, output_folder):
        self.output_folder = output_folder
        os.makedirs(output_folder, exist_ok=True)

    def save(self, image_filename, instances, img_w, img_h, meta):
        stem     = Path(image_filename).stem
        out_path = os.path.join(self.output_folder, f"{stem}.json")
        doc = {
            "image_id"         : image_filename,
            "image_resolution" : {"width":img_w,"height":img_h},
            "coord_format"     : "ymin_xmin_ymax_xmax_0to1000",
            "global_scene_context": {
                "lighting_condition": meta["lighting"],
                "water_body_type"   : meta["water_body_type"],
                "water_color"       : meta["water_color"],
                "surface_coverage"  : meta["surface_coverage"],
                "turbidity_score"   : meta["turbidity_score"],
                "annotator_notes"   : meta["annotator_notes"],
            },
            "grounded_instances": instances,
            "total_objects"     : len(instances),
            "environmental_diagnostics": {
                "primary_pollutant"   : derive_primary_pollutant(instances),
                "remediation_priority": meta["remediation_priority"],
            }
        }
        with open(out_path,"w",encoding="utf-8") as f:
            json.dump(doc, f, indent=2, ensure_ascii=False)
        return out_path

    def load_existing(self, image_filename):
        stem = Path(image_filename).stem
        path = os.path.join(self.output_folder, f"{stem}.json")
        if not os.path.exists(path):
            return None
        try:
            return json.load(open(path, encoding="utf-8"))
        except Exception:
            return None


# ══════════════════════════════════════════════════════════════════════════════
# MODULE D — CANVAS HTML + JS  v7.1
# ══════════════════════════════════════════════════════════════════════════════
CANVAS_HTML = """
<style>
  @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&display=swap');
  :root {{
    --bg:#0b0d12; --surface:rgba(26, 29, 39, 0.75); --accent:#00e5ff;
    --warn:#ff4081; --text:#e8eaf6; --muted:#9aa3c4;
    --radius:14px; --font:'Poppins', 'Helvetica Neue', Arial, sans-serif;
  }}
  body {{ background:var(--bg); margin:0; font-family:var(--font); color:var(--text); font-size:15px; }}
  #wrapper {{
    display:flex; flex-direction:column; align-items:center;
    padding:26px; gap:14px; position:relative; background:var(--surface);
    backdrop-filter: blur(12px); border-radius: var(--radius);
    border: 1px solid rgba(255, 255, 255, 0.08); box-shadow: 0 10px 34px rgba(0,0,0,0.55);
  }}
  #header  {{ font-size:15px; color:var(--muted); letter-spacing:2px; text-transform:uppercase; font-weight:600; }}
  #imgName {{ color:var(--accent); font-weight:700; font-size:20px; margin-bottom: 4px; }}

  #countBar {{
    display:flex; flex-wrap:wrap; gap:9px;
    width:100%; max-width:1000px; min-height:34px; padding:4px 0;
  }}
  .cpill {{
    display:inline-flex; align-items:center; gap:7px;
    padding:7px 16px; border-radius:22px; font-size:14px; font-weight:600;
    border:1px solid; opacity:0.97; background: rgba(255,255,255,0.06);
  }}
  .cpill-dot {{ width:11px; height:11px; border-radius:50%; flex-shrink:0; }}

  /* — bigger, higher-contrast progress bar — */
  #progressWrap {{
    width:100%; max-width:1000px; background:rgba(0,0,0,0.45); border-radius:10px;
    height:20px; border: 1px solid rgba(255,255,255,0.08); overflow: hidden; margin-top: 6px;
  }}
  #progressBar {{
    height:100%; border-radius:10px 0 0 10px;
    background:linear-gradient(90deg,#00e5ff,#1de9b6);
    box-shadow: 0 0 16px rgba(0,229,255,0.6);
    display:flex; align-items:center; justify-content:flex-end;
    transition: width 0.3s ease;
  }}
  #progressBar span {{ font-size:12px; font-weight:700; color:#04121a; padding-right:8px; white-space:nowrap; }}
  #progressLabel{{ font-size:15px; color:var(--text); text-align:right; width:100%; max-width:1000px; font-weight:600; }}

  #meta {{ display:flex; gap:26px; font-size:15px; color:var(--muted); align-items: center; margin-bottom: 4px; }}
  #lighting-badge {{ color:#ffd54f; font-weight:700; }}
  #shortcutHint {{ color:#8a94b5; font-size:13px; }}
  #shortcutHint b {{ color:#00e5ff; }}

  #canvasWrap {{
    position:relative; border:2px solid rgba(0, 229, 255, 0.5);
    box-shadow:0 0 28px rgba(0,229,255,.22); border-radius:var(--radius); overflow:hidden;
  }}
  canvas {{ display:block; border-radius:var(--radius); cursor:crosshair;
    -webkit-user-select:none; user-select:none; -webkit-touch-callout:none; touch-action:none; }}
  body, #wrapper {{ -webkit-user-select:none; user-select:none; }}
  #instructions {{ font-size:13px; color:var(--muted); text-align:center; margin-top: 8px; }}

  #classPicker {{
    position:fixed; z-index:9999; background:rgba(20, 24, 34, 0.97); backdrop-filter: blur(15px);
    border:1px solid rgba(255,255,255,0.15);
    border-radius:14px; padding:14px; box-shadow:0 15px 44px rgba(0,0,0,0.9);
    display:none; width:300px; max-height:520px; overflow-y:auto;
  }}
  #classPicker .cp-title {{
    font-size:13px; color:var(--muted); text-transform:uppercase; font-weight: 600;
    letter-spacing:1px; margin-bottom:12px; display:flex; justify-content:space-between; align-items:center;
  }}
  #classPicker .cp-cancel {{ cursor:pointer; color:#ff4081; font-size:18px; font-weight:bold; padding:0 5px; line-height:1; }}
  #classPicker .cp-cancel:hover {{ color:#ff80ab; }}
  .cp-btn {{
    display:flex; align-items:center; gap:12px; width:100%; padding:12px 14px; margin-bottom:7px;
    background:rgba(255,255,255,0.05); border:1px solid rgba(255,255,255,0.08);
    border-radius:10px; cursor:pointer; font-family:var(--font);
    font-size:15px; font-weight:500; color:#e8eaf6; text-align:left; transition:all 0.15s ease;
  }}
  .cp-btn:hover, .cp-btn.cp-hovered {{ background:rgba(255,255,255,0.16); border-color:rgba(255,255,255,0.35); transform: translateX(4px); }}
  .cp-btn.cp-last-used {{ border-color:#ffd54f88; box-shadow: 0 0 0 1px #ffd54f44 inset; }}
  .cp-dot {{ width:15px; height:15px; border-radius:50%; flex-shrink:0; }}
  .cp-key {{
    margin-left:auto; font-size:12px; font-weight:700; color:#0b0d12; background:#00e5ff;
    border-radius:6px; padding:2px 7px; min-width:16px; text-align:center;
  }}
  .cp-star {{ color:#ffd54f; font-size:12px; margin-left:4px; }}
</style>
<div id="wrapper">
  <div id="header">◈ AquaDebris Annotation Engine</div>
  <div id="imgName">{fname}</div>
  <div id="countBar">{count_pills_html}</div>
  <div id="progressWrap"><div id="progressBar" style="width:{progress_pct}%"><span>{progress_pct}%</span></div></div>
  <div id="progressLabel">Image {cur} of {total}</div>
  <div id="meta">
    <span>💡 <span id="lighting-badge">{lighting}</span></span>
    <span>Boxes: <b style="color:#00e5ff">{obj_count}</b></span>
    <span id="shortcutHint">Hold <b>Shift</b> while dragging = reuse last class &nbsp;|&nbsp; <b>Esc</b> cancel &nbsp;|&nbsp; <b>Enter</b> save &amp; next &nbsp;|&nbsp; <b>U</b> undo</span>
  </div>
  <div id="canvasWrap">
    <canvas id="ac" width="{w}" height="{h}"></canvas>
  </div>
  <div id="instructions">🖱 Click &amp; drag on the image to annotate — the class picker opens automatically</div>
</div>
<div id="classPicker">
  <div class="cp-title">
    <span>Pick a class</span>
    <span class="cp-cancel" id="cpCancel">✕</span>
  </div>
  <div id="cpButtons"></div>
</div>
<script>
(function(){{
  const canvas = document.getElementById('ac');
  const ctx    = canvas.getContext('2d');
  const imgEl  = new Image();
  imgEl.src    = 'data:image/jpeg;base64,{b64}';
  const W={w}, H={h};
  const boxes   = {confirmed_js};
  const LEGEND  = {legend_js};
  const CLASSES = {classes_js};
  const LAST_CLASS = {last_class_json};
  let sx=0,sy=0,ex=0,ey=0,drawing=false;
  let currentColor = LAST_CLASS ? (CLASSES.find(c=>c.key===LAST_CLASS)||{{}}).color || '#00e5ff'
                                  : (CLASSES.length ? CLASSES[0].color : '#00e5ff');

  let pickerOpen = false;
  let pendingBox = null;
  let pickingColor = false;

  window.activateEyedropper = function() {{ pickingColor = true; }};

  function isTyping(){{
    const el = document.activeElement;
    return el && (el.tagName === 'TEXTAREA' || el.tagName === 'INPUT');
  }}

  function drawLabel(text, x, y, color){{
    ctx.font = 'bold 13px Poppins, sans-serif';
    const tw = ctx.measureText(text).width;
    ctx.fillStyle = 'rgba(0,0,0,0.8)';
    ctx.fillRect(x, y-16, tw+8, 21);
    ctx.fillStyle = color;
    ctx.fillText(text, x+4, y);
  }}

  function drawLegend(){{
    if(!LEGEND.length) return;
    const lx=12, ly=H-12-LEGEND.length*24;
    ctx.fillStyle='rgba(15,17,23,0.78)';
    ctx.fillRect(lx-6,ly-6,175,LEGEND.length*24+12);
    LEGEND.forEach((item,i)=>{{
      const y=ly+i*24+16;
      ctx.fillStyle=item.color; ctx.fillRect(lx,y-13,15,15);
      ctx.fillStyle='#e8eaf6'; ctx.font='13px Poppins, sans-serif';
      ctx.fillText(item.label,lx+23,y);
    }});
  }}

  function redraw(){{
    ctx.drawImage(imgEl,0,0);
    boxes.forEach((b,i)=>{{
      const col=b.color||'#00e5ff';
      ctx.strokeStyle=col; ctx.lineWidth=2;
      ctx.strokeRect(b.px,b.py,b.pw,b.ph);
      ctx.fillStyle=col+'1f'; ctx.fillRect(b.px,b.py,b.pw,b.ph);
      drawLabel('#'+(i+1)+' '+b.label,b.px+4,b.py+19,col);
    }});
    if(pendingBox){{
      ctx.strokeStyle='#ffffffaa'; ctx.lineWidth=2;
      ctx.setLineDash([5,3]);
      ctx.strokeRect(pendingBox.px,pendingBox.py,pendingBox.pw,pendingBox.ph);
      ctx.setLineDash([]);
    }}
    drawLegend();
  }}
  imgEl.onload = ()=> redraw();

  const picker   = document.getElementById('classPicker');
  const cpBtns   = document.getElementById('cpButtons');
  const cpCancel = document.getElementById('cpCancel');
  const cpBtnEls = {{}};

  function chooseClass(key){{
    if(!pendingBox) return;
    const box = {{...pendingBox}};
    closePicker();
    google.colab.kernel.invokeFunction('notebook.bbox_callback',[JSON.stringify({{...box, chosen_class: key}})],{{}});
  }}

  CLASSES.forEach(cls=>{{
    const btn = document.createElement('button');
    btn.className = 'cp-btn' + (cls.key===LAST_CLASS ? ' cp-last-used' : '');
    const star = cls.key===LAST_CLASS ? '<span class="cp-star">★ last used</span>' : '';
    const keyBadge = cls.shortcut ? '<span class="cp-key">'+cls.shortcut.toUpperCase()+'</span>' : '';
    btn.innerHTML = '<span class="cp-dot" style="background:'+cls.color+'"></span><span>'+cls.display+'</span>'+star+keyBadge;
    btn.onclick = ()=> chooseClass(cls.key);
    cpBtns.appendChild(btn);
    if(cls.shortcut) cpBtnEls[cls.shortcut] = btn;
  }});

  cpCancel.onclick = ()=>{{
    const hadBox = !!pendingBox;
    pendingBox = null;
    closePicker();
    if(hadBox) google.colab.kernel.invokeFunction('notebook.cancel_callback',['cancel'],{{}});
  }};

  function openPicker(screenX, screenY){{
    pickerOpen = true;
    canvas.style.pointerEvents = 'none';
    const pw = 300, margin = 15;
    let left = screenX + margin;
    let top  = screenY;
    if(left + pw > window.innerWidth - 10) left = screenX - pw - margin;
    if(left < 5) left = 5;
    if(top + 500 > window.innerHeight) top = Math.max(5, screenY - 380);
    picker.style.left    = left + 'px';
    picker.style.top     = top  + 'px';
    picker.style.display = 'block';
  }}

  function closePicker(){{
    pickerOpen = false;
    picker.style.display = 'none';
    canvas.style.pointerEvents = 'auto';
    redraw();
  }}

  let rafPending = false;
  function scheduleDragRedraw(){{
    if(rafPending) return;
    rafPending = true;
    requestAnimationFrame(()=>{{
      rafPending = false;
      redraw();
      ctx.strokeStyle=currentColor; ctx.lineWidth=2;
      ctx.strokeRect(sx,sy,ex-sx,ey-sy);
      ctx.fillStyle=currentColor+'1a'; ctx.fillRect(sx,sy,ex-sx,ey-sy);
    }});
  }}

  canvas.onmousedown = e => {{
    if(pickerOpen) return;
    if(pickingColor) {{
      e.preventDefault();
      const r = canvas.getBoundingClientRect();
      const x = Math.round(e.clientX - r.left);
      const y = Math.round(e.clientY - r.top);
      const p = ctx.getImageData(x, y, 1, 1).data;
      const hex = "#" + ("000000" + ((1 << 24) + (p[0] << 16) + (p[1] << 8) + p[2]).toString(16)).slice(-6);
      google.colab.kernel.invokeFunction('notebook.color_picked', [hex], {{}});
      pickingColor = false;
      return;
    }}
    e.preventDefault();
    const r=canvas.getBoundingClientRect();
    sx=e.clientX-r.left; sy=e.clientY-r.top; drawing=true;
  }};

  canvas.onmousemove = e => {{
    if(!drawing||pickerOpen||pickingColor) return;
    e.preventDefault();
    const r=canvas.getBoundingClientRect();
    ex=e.clientX-r.left; ey=e.clientY-r.top;
    scheduleDragRedraw();
  }};

  canvas.onmouseup = e => {{
    if(!drawing||pickerOpen||pickingColor) return;
    e.preventDefault();
    drawing=false;
    const r=canvas.getBoundingClientRect();
    ex=e.clientX-r.left; ey=e.clientY-r.top;
    if(Math.abs(ex-sx)<15||Math.abs(ey-sy)<15){{ redraw(); return; }}

    const ymin=Math.round(Math.min(sy,ey)/H*1000);
    const xmin=Math.round(Math.min(sx,ex)/W*1000);
    const ymax=Math.round(Math.max(sy,ey)/H*1000);
    const xmax=Math.round(Math.max(sx,ex)/W*1000);

    pendingBox = {{
      bbox:[ymin,xmin,ymax,xmax],
      px:Math.min(sx,ex), py:Math.min(sy,ey),
      pw:Math.abs(ex-sx), ph:Math.abs(ey-sy)
    }};

    // Shift+drag = instantly reuse the last class, skip the picker entirely.
    if(e.shiftKey && LAST_CLASS){{
      redraw();
      chooseClass(LAST_CLASS);
      return;
    }}

    redraw();
    openPicker(e.clientX, e.clientY);
  }};

  document.addEventListener('keydown', e=>{{
    if(isTyping()) return; // never hijack keys while the user is typing Notes etc.

    if(pickerOpen){{
      const k = e.key.toLowerCase();
      if(cpBtnEls[k]){{
        e.preventDefault();
        chooseClass(CLASSES.find(c=>c.shortcut===k).key);
      }} else if(e.code==='Escape'){{
        pendingBox = null;
        closePicker();
        google.colab.kernel.invokeFunction('notebook.cancel_callback',['cancel'],{{}});
      }}
      return;
    }}

    if(pickingColor){{
      if(e.code==='Escape') pickingColor = false;
      return;
    }}

    if(e.code==='Escape'){{
      drawing=false; redraw();
    }} else if(e.code==='Enter'){{
      e.preventDefault();
      google.colab.kernel.invokeFunction('notebook.next_callback', [], {{}});
    }} else if(e.key.toLowerCase()==='u'){{
      e.preventDefault();
      google.colab.kernel.invokeFunction('notebook.undo_callback', [], {{}});
    }}
  }});
}})();
</script>"""

# ══════════════════════════════════════════════════════════════════════════════
# MODULE E — ANNOTATION SESSION  v7.1
# ══════════════════════════════════════════════════════════════════════════════
class AnnotationSession:
    def __init__(self):
        self.loader       = ImageLoader(IMAGE_FOLDER, IMAGE_LIST)
        self.exporter     = JSONExporter(JSON_FOLDER)
        self.idx          = 0
        self.instances    = []
        self.pending_bbox = None
        self.fname        = ""
        self.orig_w = self.orig_h = self.disp_w = self.disp_h = 0
        self._cached_b64  = ""
        self.last_class   = None   # persists across boxes/images for Shift+drag & picker highlight

        layout_large = widgets.Layout(width='420px', height='38px')

        # --- Metadata widgets: values PERSIST across images (sensible defaults set once) ---
        self.w_lighting = widgets.Dropdown(options=LIGHTING_OPTIONS, value="Overcast",
                                            description='💡 Lighting:', layout=layout_large)
        self.w_water_body = widgets.Dropdown(options=WATER_BODY_OPTIONS, value="Stagnant Pond",
                                              description='🌊 Water:', layout=layout_large)
        self.w_coverage = widgets.Dropdown(options=COVERAGE_OPTIONS, value="0-25%",
                                            description='📏 Coverage:', layout=layout_large)

        self.water_color_hex = "#7a5c3d"
        self.btn_pick_color = widgets.Button(description='💧 Pick Color', layout=widgets.Layout(width='130px', height='38px'))
        self.w_color_swatch = widgets.HTML(value="")
        self._update_color_swatch()
        self.w_water_color_ui = widgets.HBox([
            widgets.HTML("<div style='width:105px; color:#9aa3c4; font-family:\"Poppins\",sans-serif; font-size:15px; margin-top:6px;'>🎨 Pond colour:</div>"),
            self.btn_pick_color, self.w_color_swatch
        ], layout=layout_large)

        self.w_turbidity = widgets.Dropdown(options=TURBIDITY_OPTIONS, value=0,
                                             description='🌫 Turbidity:', layout=widgets.Layout(display='none'))
        self.turbidity_btns = {}
        swatch_cols = []
        for lvl in range(1, 11):
            btn = widgets.Button(description="", tooltip=f"Turbidity {lvl}",
                                  layout=widgets.Layout(width='42px', height='42px',
                                                         border='2px solid #2b2f3a', padding='0px'))
            btn.style.button_color = TURBIDITY_SWATCHES[lvl]
            btn._turbidity_level = lvl
            btn.on_click(self._on_turbidity_pick)
            self.turbidity_btns[lvl] = btn
            num = widgets.HTML(f"<div style='text-align:center;color:#9aa3c4;font-size:13px;margin-top:3px;font-weight:600;'>{lvl}</div>")
            swatch_cols.append(widgets.VBox([btn, num], layout=widgets.Layout(align_items='center', margin='0 5px')))
        self.w_turbidity_row   = widgets.HBox(swatch_cols)
        self.w_turbidity_label = widgets.HTML(value="")

        self.w_remediation = widgets.Dropdown(options=REMEDIATION_OPTIONS, value="None",
                                               description='🚨 Priority:', layout=layout_large)
        self.w_notes = widgets.Textarea(placeholder='Any extra observations about this image...',
                                         description='📝 Notes:', layout=widgets.Layout(width='860px', height='70px'))

        self.w_undo = widgets.Button(description='🗑️ Undo Last', button_style='danger',
                                      layout=widgets.Layout(width='160px', height='40px'))
        self.w_next = widgets.Button(description='💾 Save & Next Image →', button_style='success',
                                      layout=widgets.Layout(width='230px', height='40px'))
        self.w_status = widgets.HTML(value="")

        self.w_jump = widgets.Combobox(placeholder='Type filename or number…', options=tuple(IMAGE_LIST),
                                        description='🔎 Jump to:', ensure_option=False, layout=layout_large)
        self.w_jump_go = widgets.Button(description='Go', button_style='warning', layout=widgets.Layout(width='90px', height='38px'))
        self.w_jump_status = widgets.HTML(value="")

        self.w_undo.on_click(self._on_undo)
        self.w_next.on_click(lambda _: self._save_and_advance())
        self.w_jump_go.on_click(self._on_jump)
        self.btn_pick_color.on_click(self._on_pick_color)
        try:
            self.w_jump.on_submit(self._on_jump)   # Enter in the jump box also works
        except Exception:
            pass

        from google.colab import output as colab_output
        colab_output.register_callback('notebook.bbox_callback',   self._on_bbox)
        colab_output.register_callback('notebook.cancel_callback', self._on_cancel)
        colab_output.register_callback('notebook.color_picked',    self._on_color_picked)
        colab_output.register_callback('notebook.next_callback',   lambda: self._save_and_advance())
        colab_output.register_callback('notebook.undo_callback',   self._on_undo)

    def _update_color_swatch(self):
        self.w_color_swatch.value = (
            f"<div style='width:34px; height:34px; background-color:{self.water_color_hex}; "
            f"border:1px solid rgba(255,255,255,0.2); border-radius:7px; margin-left:16px; "
            f"box-shadow: 0 2px 6px rgba(0,0,0,0.5);'></div>")

    def _on_pick_color(self, _):
        self.btn_pick_color.description = "💧 Click image..."
        self.btn_pick_color.button_style = 'info'
        display(Javascript("if(window.activateEyedropper) window.activateEyedropper();"))

    def _on_color_picked(self, hex_color):
        self.water_color_hex = hex_color
        self._update_color_swatch()
        self.btn_pick_color.description = "💧 Pick Color"
        self.btn_pick_color.button_style = ''

    def _count_pills_html(self):
        from collections import Counter
        counts = Counter(inst["object_name"] for inst in self.instances)
        if not counts:
            return ("<span style='font-size:14px;color:#9aa3c4'>"
                    "No objects yet — draw a box directly on the image to start</span>")
        pills = []
        for cls_key, cnt in counts.items():
            col  = CLASS_COLORS.get(cls_key, DEFAULT_COLOR)
            name = TAXONOMY[cls_key]["display"] if cls_key in TAXONOMY else cls_key
            pills.append(
                f"<span class='cpill' style='border-color:{col}55;color:{col}'>"
                f"<span class='cpill-dot' style='background:{col}'></span>"
                f"{name}&nbsp;<b>{cnt}</b></span>"
            )
        return "".join(pills)

    def _render_canvas(self):
        confirmed_js = json.dumps([{
            "label": inst["object_name"],
            "color": CLASS_COLORS.get(inst["object_name"], DEFAULT_COLOR),
            "px": inst["_display"]["px"], "py": inst["_display"]["py"],
            "pw": inst["_display"]["pw"], "ph": inst["_display"]["ph"]
        } for inst in self.instances])

        used = list(dict.fromkeys(inst["object_name"] for inst in self.instances))
        legend_js = json.dumps([{"label": k, "color": CLASS_COLORS.get(k, DEFAULT_COLOR)} for k in used])
        progress_pct = round(self.idx / max(len(IMAGE_LIST), 1) * 100)

        html = CANVAS_HTML.format(
            fname            = self.fname,
            cur              = self.idx + 1,
            total            = len(IMAGE_LIST),
            progress_pct     = progress_pct,
            lighting         = self.w_lighting.value or "—",
            obj_count        = len(self.instances),
            w                = self.disp_w,
            h                = self.disp_h,
            b64              = self._cached_b64,
            confirmed_js     = confirmed_js,
            legend_js        = legend_js,
            classes_js       = CLASSES_JS,
            last_class_json  = json.dumps(self.last_class),
            count_pills_html = self._count_pills_html(),
        )
        clear_output(wait=True)
        display(HTML(html))

    def _load_image(self):
        """Load the current image. Metadata dropdowns are left untouched (they
        persist from the previous image) UNLESS this image already has a saved
        JSON, in which case that JSON's values take over so re-editing is safe."""
        self.pending_bbox = None
        self.w_notes.value = ""
        self.instances = []
        self.w_jump_status.value = ""

        self.fname, self._cached_b64, self.disp_w, self.disp_h, self.orig_w, self.orig_h = self.loader.load(self.idx)

        existing = self.exporter.load_existing(self.fname)
        if existing:
            ctx  = existing.get("global_scene_context", {})
            diag = existing.get("environmental_diagnostics", {})
            self.w_lighting.value    = ctx.get("lighting_condition", self.w_lighting.value)
            self.w_water_body.value  = ctx.get("water_body_type", self.w_water_body.value)

            _wc = ctx.get("water_color") or self.water_color_hex
            if isinstance(_wc, str) and _wc.startswith("#"):
                self.water_color_hex = _wc
            self._update_color_swatch()

            self.w_coverage.value    = ctx.get("surface_coverage", self.w_coverage.value)
            self.w_turbidity.value   = ctx.get("turbidity_score", self.w_turbidity.value)
            self.w_remediation.value = diag.get("remediation_priority", self.w_remediation.value)
            self.w_notes.value       = ctx.get("annotator_notes", "")
            for inst in existing.get("grounded_instances", []):
                ymin, xmin, ymax, xmax = inst["bbox_2d"]
                px = round(xmin/1000 * self.disp_w)
                py = round(ymin/1000 * self.disp_h)
                pw = round((xmax-xmin)/1000 * self.disp_w)
                ph = round((ymax-ymin)/1000 * self.disp_h)
                self.instances.append({
                    "instance_id": inst["instance_id"],
                    "object_name": inst["object_name"],
                    "bbox_2d"    : inst["bbox_2d"],
                    "fcot"       : inst["fcot"],
                    "_display"   : {"px":px,"py":py,"pw":pw,"ph":ph}
                })
            if self.instances:
                self.last_class = self.instances[-1]["object_name"]

        self._refresh_turbidity_selection()
        self._render_setup_panel()

    def _render_setup_panel(self):
        self._render_canvas()   # canvas is active immediately — no lock, no confirm step
        display(widgets.HBox([self.w_jump, self.w_jump_go]))
        if self.w_jump_status.value:
            display(self.w_jump_status)

        display(widgets.VBox([
            widgets.HTML("<div style='font-family:Poppins;color:#9aa3c4;font-size:15px;margin-bottom:8px'>📋 Environmental Metadata (carries over from the last image automatically — only change what's different):</div>"),
            widgets.HBox([self.w_lighting, self.w_water_body]),
            widgets.HBox([self.w_water_color_ui, self.w_coverage]),
            widgets.HTML("<b style='color:#b0bec5;font-size:14px; margin-top:8px; display:block;'>🌫 Turbidity — tap the swatch matching the pond colour:</b>"),
            self.w_turbidity_row,
            self.w_turbidity_label,
            self.w_remediation,
            self.w_notes,
        ]))
        self._show_action_bar()

    def _on_turbidity_pick(self, btn):
        self.w_turbidity.value = btn._turbidity_level
        self._refresh_turbidity_selection()

    def _refresh_turbidity_selection(self):
        lvl = self.w_turbidity.value
        for l, b in self.turbidity_btns.items():
            b.layout.border = "3px solid #00e5ff" if l == lvl else "2px solid rgba(255,255,255,0.1)"
        if not lvl:
            self.w_turbidity_label.value = "<span style='color:#9aa3c4;font-size:14px'>Not set — tap a swatch above.</span>"
        else:
            hexc = TURBIDITY_SWATCHES.get(lvl, "#888")
            cap  = TURBIDITY_CAPTIONS.get(lvl, "")
            self.w_turbidity_label.value = (
                f"<span style='color:#00e5ff;font-size:14px'>Selected: <b>{lvl}/10</b> "
                f"<span style='display:inline-block;width:15px;height:15px;background:{hexc};border:1px solid #555;vertical-align:middle;border-radius:3px;margin:0 6px'></span>{cap}</span>")

    def _show_action_bar(self):
        display(widgets.VBox([
            widgets.HTML("<hr style='border:1px solid rgba(255,255,255,0.1); width:100%; max-width:1000px;'>"
                         "<span style='color:#9aa3c4;font-size:14px'>One click saves this image and moves to the next. "
                         "Keyboard: <b style='color:#00e5ff'>Shift</b>+drag = reuse last class, "
                         "<b style='color:#00e5ff'>1-9/0/O</b> = pick class, "
                         "<b style='color:#00e5ff'>Enter</b> = save &amp; next, "
                         "<b style='color:#00e5ff'>U</b> = undo.</span>"),
            widgets.HBox([self.w_next, self.w_undo]),
            self.w_status,
        ]))

    def _on_bbox(self, msg_str):
        data = json.loads(msg_str)
        self.pending_bbox = {
            "bbox"    : data["bbox"],
            "_display": {"px":data["px"],"py":data["py"], "pw":data["pw"],"ph":data["ph"]}
        }
        chosen = data.get("chosen_class")
        if not chosen: return

        if chosen == "__others__":
            self._show_custom_input_panel()
            return

        fcot = TAXONOMY[chosen]["fcot"]
        self.instances.append({
            "instance_id": len(self.instances)+1,
            "object_name": chosen,
            "bbox_2d"    : self.pending_bbox["bbox"],
            "fcot"       : fcot,
            "_display"   : self.pending_bbox["_display"]
        })
        self.last_class = chosen
        self.pending_bbox = None
        col = CLASS_COLORS.get(chosen, DEFAULT_COLOR)
        self.w_status.value = f"<span style='color:{col}; font-size:15px;'>✅ <b>{chosen}</b> added (total: {len(self.instances)})</span>"
        self._render_setup_panel()

    def _on_cancel(self, _=None):
        self.pending_bbox = None
        self.w_status.value = "<span style='color:#9aa3c4; font-size:15px;'>↩ Cancelled Box.</span>"
        self._render_setup_panel()

    def _show_custom_input_panel(self):
        w_name = widgets.Text(placeholder="Enter custom class name", description="Name:", layout=widgets.Layout(width="420px", height='38px'))
        w_add  = widgets.Button(description="➕ Add", button_style="info", layout=widgets.Layout(width="130px", height='38px'))
        def _add(_):
            name = w_name.value.strip()
            if not name: return
            self.instances.append({
                "instance_id": len(self.instances)+1,
                "object_name": name,
                "bbox_2d"    : self.pending_bbox["bbox"],
                "fcot"       : TAXONOMY["__others__"]["fcot"],
                "_display"   : self.pending_bbox["_display"]
            })
            self.last_class = name
            self.pending_bbox = None
            self.w_status.value = f"<span style='color:#b0bec5; font-size:15px;'>✅ Added: <b>{name}</b></span>"
            self._render_setup_panel()
        w_add.on_click(_add)
        display(widgets.VBox([
            widgets.HTML("<b style='color:#b0bec5; font-size:15px;'>✏️ Custom class name:</b>"),
            widgets.HBox([w_name, w_add]),
        ]))

    def _on_undo(self, _=None):
        if not self.instances:
            self.w_status.value = "<span style='color:#ff4081; font-size:15px;'>⚠ Nothing to undo.</span>"
            self._render_setup_panel(); return
        removed = self.instances.pop()
        for i, inst in enumerate(self.instances): inst["instance_id"] = i+1
        self.w_status.value = (f"<span style='color:#ffd54f; font-size:15px;'>↩ Removed: <b>{removed['object_name']}</b> "
                                f"(remaining: {len(self.instances)})</span>")
        self._render_setup_panel()

    def _on_jump(self, _=None):
        query = self.w_jump.value.strip()
        if not query:
            return

        target_idx = None
        ambiguous  = None

        if query in IMAGE_LIST:
            target_idx = IMAGE_LIST.index(query)
        else:
            if query.isdigit():
                n = int(query)
                if 1 <= n <= len(IMAGE_LIST):
                    target_idx = n - 1
            if target_idx is None:
                matches = [i for i, f in enumerate(IMAGE_LIST) if query.lower() in f.lower()]
                if len(matches) == 1:
                    target_idx = matches[0]
                elif len(matches) > 1:
                    ambiguous = matches

        if target_idx is None:
            if ambiguous is not None:
                names = ", ".join(IMAGE_LIST[i] for i in ambiguous[:5])
                more  = "…" if len(ambiguous) > 5 else ""
                self.w_jump_status.value = (f"<span style='color:#ff4081'>⚠ {len(ambiguous)} files match "
                                            f"'<b>{query}</b>': {names}{more}. Be more specific.</span>")
            else:
                self.w_jump_status.value = (f"<span style='color:#ff4081'>⚠ No match for "
                                            f"'<b>{query}</b>'. Try the exact filename or an image number "
                                            f"(1-{len(IMAGE_LIST)}).</span>")
            self._render_setup_panel()
            return

        self.idx = target_idx
        self.w_jump.value = ""
        self._load_image()

    def _save_and_advance(self):
        """Single-click (or Enter) save & advance — no confirmation dialog."""
        clean = [{"instance_id":i["instance_id"],"object_name":i["object_name"],
                  "bbox_2d":i["bbox_2d"],"fcot":i["fcot"]}
                 for i in self.instances]
        meta  = {
            "lighting"            : self.w_lighting.value,
            "water_body_type"     : self.w_water_body.value,
            "water_color"         : self.water_color_hex,
            "surface_coverage"    : self.w_coverage.value,
            "turbidity_score"     : self.w_turbidity.value,
            "annotator_notes"     : self.w_notes.value.strip(),
            "remediation_priority": self.w_remediation.value,
        }
        path = self.exporter.save(self.fname, clean, self.orig_w, self.orig_h, meta)
        print(f"✅  Saved ({len(clean)} objects): {path}")
        self.idx += 1
        self._advance_to_next_pending()
        if self.idx < len(IMAGE_LIST):
            self._load_image()
        else:
            clear_output(wait=True); print("🎉  All images annotated!")

    def _already_done(self, fname):
        stem = Path(fname).stem
        path = os.path.join(JSON_FOLDER, f"{stem}.json")
        if not os.path.exists(path): return False
        try:
            doc = json.load(open(path))
            return "image_id" in doc and "grounded_instances" in doc
        except Exception:
            print(f"⚠  Corrupt JSON for {fname} — will re-annotate."); return False

    def _advance_to_next_pending(self):
        while self.idx < len(IMAGE_LIST) and self._already_done(IMAGE_LIST[self.idx]):
            print(f"⏭  Skipping (done): {IMAGE_LIST[self.idx]}"); self.idx += 1

    def start(self):
        if not IMAGE_LIST:
            print("❌  No images found. Check IMAGE_FOLDER in Block 2."); return
        done = [f for f in IMAGE_LIST if self._already_done(f)]
        todo = [f for f in IMAGE_LIST if not self._already_done(f)]
        print(f"📊  Total : {len(IMAGE_LIST)}")
        print(f"✅  Done  : {len(done)}")
        print(f"🔲  To-do : {len(todo)}")
        if done:
            print(f"   Skipping: {', '.join(done[:8])}{'...' if len(done)>8 else ''}")
        print()
        self._advance_to_next_pending()
        if self.idx >= len(IMAGE_LIST):
            print("🎉  All images already annotated!"); return
        self._load_image()

# ══════════════════════════════════════════════════════════════════════════════
# RUN
# ══════════════════════════════════════════════════════════════════════════════
session = AnnotationSession()
session.start()